In [ ]:
#!/usr/bin/env python
# coding: utf-8

import os
import time
import ee
import pandas as pd

# -------------------- 0) EE AUTH (service account, high-volume) --------------------
SERVICE_ACCOUNT = 'gee-serdp-upload@appspot.gserviceaccount.com'
SA_KEY_PATH     = '/explore/nobackup/people/spotter5/cnn_mapping/gee-serdp-upload-7cd81da3dc69.json'

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = SA_KEY_PATH
print("Initializing Earth Engine via service account + high-volume endpoint...")
credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, SA_KEY_PATH)
ee.Initialize(credentials)
ee.Initialize(credentials, opt_url='https://earthengine-highvolume.googleapis.com')
print("EE initialized.\n")

# -------------------- 1) I/O PATHS --------------------
input_training_data_path  = '/explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_data_v5.csv'
output_embeddings_path    = '/explore/nobackup/people/spotter5/anna_v/v2/satellite_embeddings_1km.csv'

print(f"Loading source data from: {input_training_data_path}")
df_source = pd.read_csv(input_training_data_path, usecols=['site_reference', 'latitude', 'longitude', 'year'])
unique_site_years = df_source.drop_duplicates(subset=['site_reference', 'year']).reset_index(drop=True)
print(f"Found {len(unique_site_years)} unique site-year combos.\n")

# -------------------- 2) EMBEDDING COLLECTION --------------------
COL_ID = 'GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL'
se_collection = ee.ImageCollection(COL_ID)

# -------------------- 3) Helper: get 1 km-mean version of the annual image --------------------
def embed_image_1km_mean(img: ee.Image) -> ee.Image:
    """
    Aggregate each band to 1 km by taking the mean of native ~10 m pixels.
    Use an INT32-safe maxPixels for reduceResolution.
    """
    base_proj = img.projection()
    agg = img.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=16384   # <-- CHANGED: < 65536 and safely above ~10,000 px per km
    )
    # Force the 1 km reprojection
    return agg.reproject(crs=base_proj, scale=1000)

# -------------------- 4) Extraction Loop --------------------
results = []
total = len(unique_site_years)
print("Starting GEE data extraction at 1 km…\n")

for idx, row in unique_site_years.iterrows():
    site = row['site_reference']
    lat  = float(row['latitude'])
    lon  = float(row['longitude'])
    orig_year = int(row['year'])
    yr = 2017 if orig_year < 2017 else orig_year

    pt = ee.Geometry.Point([lon, lat])
    start = f'{yr}-01-01'
    end   = f'{yr+1}-01-01'

    try:
        img = se_collection.filterBounds(pt).filterDate(start, end).first()
        band_names = ee.Image(img).bandNames().getInfo()
        if not band_names or len(band_names) < 64:
            print(f"  - WARNING: No/partial image for {site} ({orig_year}→{yr}). Skipping.")
            continue

        img_1km = embed_image_1km_mean(img)

        emb_dict = img_1km.reduceRegion(
            reducer   = ee.Reducer.first(),
            geometry  = pt,
            scale     = 1000,
            maxPixels = 65535,   # <-- CHANGED: < 65536 for point sampling
            tileScale = 2
        ).getInfo()

        if not emb_dict:
            print(f"  - WARNING: Empty reduction for {site} ({orig_year}→{yr}).")
            continue

        out_row = {'site_reference': site, 'year': orig_year}
        for i in range(64):
            a = f'A{i:02d}'
            out_row[f'SE_{i}'] = emb_dict.get(a, None)
        results.append(out_row)

    except Exception as e:
        print(f"  - ERROR for {site} ({orig_year}→{yr}): {e}")
        time.sleep(3)

    if (idx + 1) % 25 == 0 or (idx + 1) == total:
        print(f"Processed {idx + 1} / {total}")

print(f"\nExtraction complete. Rows gathered: {len(results)}")

# -------------------- 5) Save --------------------
if results:
    df_out = pd.DataFrame(results)
    df_out.to_csv(output_embeddings_path, index=False)
    print(f"Saved 1 km embeddings to: {output_embeddings_path}")
    print(df_out.head())
else:
    print("No results to save.")


Initializing Earth Engine via service account + high-volume endpoint...
EE initialized.

Loading source data from: /explore/nobackup/people/spotter5/anna_v/v2/v2_model_training_data_v5.csv
Found 49766 unique site-year combos.

Starting GEE data extraction at 1 km…

Processed 25 / 49766
Processed 50 / 49766
Processed 75 / 49766
Processed 100 / 49766
Processed 125 / 49766
Processed 150 / 49766
Processed 175 / 49766
Processed 200 / 49766
Processed 225 / 49766
Processed 250 / 49766
Processed 275 / 49766
Processed 300 / 49766
Processed 325 / 49766
Processed 350 / 49766
Processed 375 / 49766
Processed 400 / 49766
Processed 425 / 49766
Processed 450 / 49766
Processed 475 / 49766
Processed 500 / 49766
Processed 525 / 49766
Processed 550 / 49766
Processed 575 / 49766
Processed 600 / 49766
Processed 625 / 49766
Processed 650 / 49766
Processed 675 / 49766
Processed 700 / 49766
Processed 725 / 49766
Processed 750 / 49766
Processed 775 / 49766
Processed 800 / 49766
Processed 825 / 49766
Processed 8